In [15]:
import os
import json
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [16]:
# rebuilding model structure
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, pool: bool = True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class SpringfieldCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  pool=True),
            nn.Dropout2d(0.1),
            ConvBlock(32,  64,  pool=True),
            nn.Dropout2d(0.1),
            ConvBlock(64,  128, pool=True),
            nn.Dropout2d(0.2),
            ConvBlock(128, 256, pool=True),
            nn.Dropout2d(0.2),
            ConvBlock(256, 256, pool=False),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

In [25]:
def infer(data_dir, model_path):
    """
    Loads the trained model and predicts the class for every image in data_dir.
    Saves results to results.json in the current directory.

    Parameters
    ----------
    data_dir   : str | Path  – folder containing images to classify
    model_path : str | Path  – path to the saved model.pth checkpoint

    Output
    ------
    results.json
        {
            "pic_999.jpg":  "homer_simpson",
            "pic_1000.jpg": "marge_simpson",
            ...
        }
    """
    data_dir   = Path(data_dir)
    model_path = Path(model_path)

    # Load checkpoint
    print(f'Loading checkpoint from: {model_path}')
    checkpoint = torch.load(model_path, map_location=DEVICE)

    classes     = checkpoint['classes']       # e.g. ['bart_simpson', 'homer_simpson', ...]
    num_classes = checkpoint['num_classes']
    img_size    = checkpoint['img_size']

    print(f'Classes    : {classes}')
    print(f'Num classes: {num_classes}')
    print(f'Image size : {img_size}x{img_size}')

    #  Rebuild model and load weights
    model = SpringfieldCNN(num_classes=num_classes).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()   # disables dropout / batchnorm training behaviour
    print('Model loaded and set to eval mode.')

    # Transform — must match val_transforms from training
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225]),
    ])

    # Collect all image files
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    image_paths = [
    p for p in sorted(data_dir.rglob('*'))
    if p.suffix.lower() in valid_ext
    ]
    print(f'Found {len(image_paths)} images in {data_dir}')

    # ── 5. Run inference ───────────────────────────────────────────────────
    results = {}

    with torch.no_grad():
        for img_path in image_paths:
            img    = Image.open(img_path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(DEVICE)  # (1, 3, H, W)
            logits = model(tensor)                            # (1, num_classes)
            pred_idx   = logits.argmax(dim=1).item()
            pred_class = classes[pred_idx]
            results[img_path.name] = pred_class

    # ── 6. Save results.json ───────────────────────────────────────────────
    output_path = Path('results.json')
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f'\nDone. {len(results)} predictions saved to {output_path.resolve()}')
    return results

In [26]:
DATA_DIR   = '/content/drive/MyDrive/characters_train'
MODEL_PATH = "model (1).pth"   # saved by train.ipynb

results = infer(DATA_DIR, MODEL_PATH)

Loading checkpoint from: model (1).pth
Classes    : ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler', 'comic_book_guy', 'disco_stu', 'edna_krabappel', 'fat_tony', 'gil', 'groundskeeper_willie', 'homer_simpson', 'kent_brockman', 'krusty_the_clown', 'lenny_leonard', 'lionel_hutz', 'lisa_simpson', 'maggie_simpson', 'marge_simpson', 'martin_prince', 'mayor_quimby', 'milhouse_van_houten', 'miss_hoover', 'moe_szyslak', 'ned_flanders', 'nelson_muntz', 'otto_mann', 'patty_bouvier', 'principal_skinner', 'professor_john_frink', 'rainier_wolfcastle', 'ralph_wiggum', 'selma_bouvier', 'sideshow_bob', 'sideshow_mel', 'snake_jailbird', 'troy_mcclure', 'waylon_smithers']
Num classes: 42
Image size : 64x64
Model loaded and set to eval mode.
Found 16795 images in /content/drive/MyDrive/characters_train

Done. 1828 predictions saved to /content/results.json


In [27]:
# first 10 predictions
preview = dict(list(results.items())[:10])
for filename, predicted_class in preview.items():
    print(f'{filename:30s} →  {predicted_class}')

pic_0000.jpg                   →  waylon_smithers
pic_0001.jpg                   →  waylon_smithers
pic_0002.jpg                   →  waylon_smithers
pic_0003.jpg                   →  waylon_smithers
pic_0004.jpg                   →  waylon_smithers
pic_0005.jpg                   →  waylon_smithers
pic_0006.jpg                   →  principal_skinner
pic_0007.jpg                   →  waylon_smithers
pic_0008.jpg                   →  waylon_smithers
pic_0009.jpg                   →  sideshow_bob
